In [1]:
import os
os.chdir("/home/ruslan/thesis/cl")

In [ ]:
from datasets.full_dataset import CombinedDataset

from moviad.datasets.bmad.bmad_dataset import BMAD

from memories.replay_strategy import ReplayModel
from trainers.models import STFPMModel


continual_dataset = CombinedDataset(BMAD, task_type="segmentation", root_dir="/mnt/disk1/ruslan_nuriev/bmad")

# Create model and strategy
replay_strategy = ReplayModel(
    model_conf={'stfpm': STFPMModel("cuda:0", 'resnet18', ['layer1', 'layer2', 'layer3'])},
    buffer_size=1000
)


/home/ruslan/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(train) Task 0 (liver): 1542 samples
(train) Task 1 (chest): 8000 samples
(train) Task 2 (histopathology): 5088 samples
(train) Task 3 (brain): 7500 samples
(train) Task 4 (retinaoct): 26315 samples
(train) Task 5 (retinaresc): 4297 samples


In [3]:
data_full = continual_dataset.load_categories()

In [4]:
replay_strategy._init_model()

In [5]:
from tqdm import tqdm
import numpy as np

history = []
for epoch in range(1):
    epoch_losses = []
    with tqdm(data_full, desc=f"Epoch {epoch+1}/{1}") as pbar:
        for batch in pbar:
            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch
            
            loss = replay_strategy.partial_update(images)
            epoch_losses.append(loss)
            pbar.set_postfix(loss=f"{loss:.4f}")
    
    avg_loss = np.mean(epoch_losses)
    print(f"  Epoch {epoch+1}/{1}, Loss: {avg_loss:.4f}")

Epoch 1/1: 100%|██████████| 1649/1649 [06:32<00:00,  4.20it/s, loss=0.2646]

  Epoch 1/1, Loss: 0.6096


In [9]:
from moviad.datasets.bmad.bmad_dataset import BMAD
import torch
from moviad.utilities.evaluator import Evaluator

data = BMAD("segmentation", "/mnt/disk1/ruslan_nuriev/bmad", 'liver', 'test', True, (224, 224))
liver_test = torch.utils.data.DataLoader(data, batch_size=32)
replay_strategy.model.eval()

eval = Evaluator(liver_test, "cuda:0")
eval.evaluate(replay_strategy.model)

Eval: 100%|██████████| 47/47 [00:04<00:00,  9.92it/s]


{'img_roc_auc': np.float64(0.5263960129506349),
 'pxl_roc_auc': np.float64(0.5681017668888302),
 'img_f1': np.float64(0.6145930776426568),
 'pxl_f1': np.float64(0.009531219522104414),
 'img_pr_auc': np.float64(0.46643827636389534),
 'pxl_pr_auc': np.float64(0.00438684498622257),
 'pxl_au_pro': np.float64(0.4743882642018142)}

In [7]:
1649 * 32

52768